# American option pricing experiments
One notebook for LSMC, Haugh-Kogan, Herrera, and Alfonsi experiments.

In [4]:
import gc, time
from math import comb
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

from activations import ReLU, SiLU, Softplus, tanh
from payoff import basket_call_ram, basket_put_ram, geometric_put_ram, max_call_ram, min_put_ram
from simulation_paths import simulate_gbm_multi
from LSMC.LSMC_standard import american_LSMC_standard
from LSMC.LSMC_random import american_LSMC_random
from HK_and_Herrera.HK_standard import american_HK_standard
from HK_and_Herrera.HK_random import american_HK_random
from HK_and_Herrera.Herrera_random import american_HK_Herrera_random
from ALFONSI.alfonsi_standard_local import american_AL_PRIMAL_DUAL_standard_local
from ALFONSI.alfonsi_random import american_AL_PRIMAL_DUAL

## Common market parameters

In [5]:
MATURITY = 3.0 
STRIKE = 100.0 
RATE = 0.05 
DIVIDEND = 0.10 
VOLATILITY = 0.20 
EXERCISE_DATES = 9
BASIS_SEED_OFFSET = 100_000
COMPARISON_OUTPUT_FILE = Path("results_comparison.csv")

## Shared functions

In [6]:
PAYOFFS = {"max-call": max_call_ram, "min-put": min_put_ram, "basket-call": basket_call_ram, "basket-put": basket_put_ram, "geometric-put": geometric_put_ram}
ACTIVATIONS = {"relu": ReLU, "silu": SiLU, "softplus": Softplus, "tanh": tanh}
AVAILABLE_LSMC_HK_METHODS = {"lsmc-standard", "lsmc-random", "hk-standard", "hk-random", "hk-herrera"}
AVAILABLE_ALFONSI_METHODS = {"alfonsi-standard", "alfonsi-random"}

def volatility_for(dimension, asymmetric):
    if not asymmetric: 
        return VOLATILITY
    if dimension <= 5: 
        return np.linspace(0.08, 0.40, dimension, dtype=np.float32)
    return 0.10 + np.arange(1, dimension + 1, dtype=np.float32) / (2 * dimension)

def simulate(paths, dimension, initial_price, seed, asymmetric=False, subintervals=1):
    return simulate_gbm_multi(mu=RATE - DIVIDEND, sigma=volatility_for(dimension, asymmetric), T=MATURITY, 
                dt=MATURITY / (EXERCISE_DATES * subintervals),  paths=paths, nb_stocks=dimension, sim_seed=seed, initial_prices=initial_price)

def summarize_results(results):
    columns = ["method", "payoff", "dimension", "initial_price", "seeds", "lower_mean", "lower_seed_sd", "upper_mean", "upper_seed_sd", "mean_duration_seconds"]
    if results.empty: return pd.DataFrame(columns=columns)
    completed = results[results["status"].eq("ok")]
    summary = completed.groupby(["method", "payoff", "dimension", "initial_price"], as_index=False).agg(seeds=("seed", "nunique"), lower_mean=("lower_bound", "mean"), lower_seed_sd=("lower_bound", "std"), upper_mean=("upper_bound", "mean"), upper_seed_sd=("upper_bound", "std"), mean_duration_seconds=("duration_seconds", "mean"))
    return summary[columns]

## LSMC, Haugh-Kogan, and Herrera configuration

Available methods: `lsmc-standard`, `lsmc-random`, `hk-standard`, `hk-random`, `hk-herrera`.

In [7]:
RUN_LSMC_HK = True; 
SELECTED_LSMC_HK_METHODS = ["lsmc-standard", "lsmc-random", "hk-standard", "hk-random"]
LSMC_HK_PAYOFFS = ["max-call"] 
LSMC_HK_DIMENSIONS = [5] 
LSMC_HK_INITIAL_PRICES = [90.0, 100.0, 110.0]
LSMC_HK_SEEDS = [101, 202, 303, 404, 505]
LSMC_HK_ASYMMETRIC_VOLATILITY = False 
LSMC_HK_TRAINING_PATHS = 100_000; LSMC_HK_PRICING_PATHS = 50_000
LSMC_HK_DEGREE = 2 
LSMC_HK_NEURONS = 100 
LSMC_HK_ACTIVATION = "silu" 
HERRERA_ACTIVATION = "softplus"
LSMC_HK_FEATURE_SCALE = 1.0 
LSMC_HK_RIDGE = 1e-8 
MAX_STANDARD_BASIS_ENTRIES = 100_000_000
LSMC_HK_OUTPUT_FILE = Path("results_lsmc_hk.csv")
LSMC_HK_SUMMARY_FILE = Path("summary_lsmc_hk.csv")

## Alfonsi configuration

Available methods: `alfonsi-standard`, `alfonsi-random`.

In [8]:
RUN_ALFONSI = True 
SELECTED_ALFONSI_METHODS = ["alfonsi-standard", "alfonsi-random"]
ALFONSI_PAYOFFS = ["max-call"]
ALFONSI_DIMENSIONS = [2]
ALFONSI_INITIAL_PRICES = [90.0, 100.0, 110.0] 
ALFONSI_SEEDS = [101, 202, 303, 404, 505]
ALFONSI_ASYMMETRIC_VOLATILITY = False
ALFONSI_SUBINTERVALS = 1
ALFONSI_MARTINGALE_PATHS = 1_000_000 
ALFONSI_POLICY_PATHS = 100_000 
ALFONSI_PRICING_PATHS = 100_000
ALFONSI_CELLS_PER_DIMENSION = 10 
ALFONSI_DEGREE = 2 
ALFONSI_H_REG = 100 
ALFONSI_H_MART = 80
ALFONSI_ACTIVATION = "softplus" 
ALFONSI_FEATURE_SCALE = 1.0 
ALFONSI_RIDGE = 1e-6 
MAX_LOCAL_COEFFICIENTS = 10_000_000
ALFONSI_OUTPUT_FILE = Path("results_alfonsi.csv")
ALFONSI_SUMMARY_FILE = Path("summary_alfonsi.csv")

## LSMC, Haugh-Kogan, and Herrera functions

In [9]:
def lsmc_hk_row(method, payoff_name, dimension, initial_price, seed):
    return {"method": method, "payoff": payoff_name, "dimension": dimension, "initial_price": initial_price, "seed": seed, 
    "training_paths": LSMC_HK_TRAINING_PATHS, "pricing_paths": LSMC_HK_PRICING_PATHS,  "degree": LSMC_HK_DEGREE, "neurons": LSMC_HK_NEURONS, 
    "activation": HERRERA_ACTIVATION if method == "hk-herrera" else LSMC_HK_ACTIVATION, 
    "feature_scale": LSMC_HK_FEATURE_SCALE, "ridge": LSMC_HK_RIDGE}

def evaluate_lsmc_hk(method, payoff, train_set, price_set, basis_seed):
    interval = MATURITY / EXERCISE_DATES
    if method == "lsmc-standard":
        lower, lower_se = american_LSMC_standard(payoff, STRIKE, RATE, MATURITY, interval, train_set, price_set, LSMC_HK_PRICING_PATHS, LSMC_HK_DEGREE, LSMC_HK_RIDGE)
        return lower, lower_se, None, None
    if method == "lsmc-random":
        lower, lower_se = american_LSMC_random(payoff, STRIKE, RATE, MATURITY, interval, ACTIVATIONS[LSMC_HK_ACTIVATION], train_set, price_set, LSMC_HK_FEATURE_SCALE, LSMC_HK_NEURONS, basis_seed, LSMC_HK_RIDGE)
        return lower, lower_se, None, None
    common = dict(payoff=payoff, strike_price=STRIKE, interest_rate=RATE, exercise_dates=EXERCISE_DATES, interval=interval, train_set=train_set, price_set=price_set)
    if method == "hk-standard": 
        result = american_HK_standard(**common, max_deg=LSMC_HK_DEGREE, ridge_alpha=LSMC_HK_RIDGE)
    elif method == "hk-random": 
        result = american_HK_random(**common, std_basis=LSMC_HK_FEATURE_SCALE, basis_seed=basis_seed, nneurons=LSMC_HK_NEURONS, activation=ACTIVATIONS[LSMC_HK_ACTIVATION], ridge_alpha=LSMC_HK_RIDGE)
    else: 
        result = american_HK_Herrera_random(**common, std_basis=LSMC_HK_FEATURE_SCALE, basis_seed=basis_seed, nneurons=LSMC_HK_NEURONS, activation=ACTIVATIONS[HERRERA_ACTIVATION], ridge_alpha=LSMC_HK_RIDGE)
    upper, upper_se, lower, lower_se = result

    return lower, lower_se, upper, upper_se

def run_lsmc_hk_experiments():
    rows = []
    for dimension in LSMC_HK_DIMENSIONS:
        basis_entries = comb(dimension + LSMC_HK_DEGREE, LSMC_HK_DEGREE) * LSMC_HK_TRAINING_PATHS
        for initial_price in LSMC_HK_INITIAL_PRICES:
            for seed in LSMC_HK_SEEDS:
                runnable = []
                for method in SELECTED_LSMC_HK_METHODS:
                    if method.endswith("standard") and basis_entries > MAX_STANDARD_BASIS_ENTRIES:
                        for payoff_name in LSMC_HK_PAYOFFS: 
                            rows.append({**lsmc_hk_row(method, payoff_name, dimension, initial_price, seed), "status": "skipped: polynomial basis too large"})
                    else: 
                        runnable.append(method)
                if not runnable: continue
                train_set = simulate(LSMC_HK_TRAINING_PATHS, dimension, initial_price, seed, LSMC_HK_ASYMMETRIC_VOLATILITY)
                price_set = simulate(LSMC_HK_PRICING_PATHS, dimension, initial_price, seed + 1, LSMC_HK_ASYMMETRIC_VOLATILITY)
                for payoff_name in LSMC_HK_PAYOFFS:
                    payoff = PAYOFFS[payoff_name]
                    for method in runnable:
                        started = time.perf_counter()
                        lower, lower_se, upper, upper_se = evaluate_lsmc_hk(method, payoff, train_set, price_set, BASIS_SEED_OFFSET + seed)
                        rows.append({**lsmc_hk_row(method, payoff_name, dimension, initial_price, seed), "lower_bound": lower, "lower_standard_error": lower_se, "upper_bound": upper, "upper_standard_error": upper_se, "duration_seconds": time.perf_counter() - started, "status": "ok"})
                        print(f"Completed {method}/{payoff_name}: d={dimension}, S0={initial_price:g}, seed={seed}")
                del train_set, price_set; gc.collect()
    result = pd.DataFrame(rows); LSMC_HK_OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True); result.to_csv(LSMC_HK_OUTPUT_FILE, index=False)
    return result

## Alfonsi functions

In [10]:
def alfonsi_row(method, payoff_name, dimension, initial_price, seed):
    return {"method": method, "payoff": payoff_name, "dimension": dimension, "initial_price": initial_price, "seed": seed, 
    "martingale_paths": ALFONSI_MARTINGALE_PATHS, "policy_paths": ALFONSI_POLICY_PATHS, "pricing_paths": ALFONSI_PRICING_PATHS, 
    "subintervals": ALFONSI_SUBINTERVALS, "cells_per_dimension": ALFONSI_CELLS_PER_DIMENSION, "degree": ALFONSI_DEGREE, 
    "h_reg": ALFONSI_H_REG, "h_mart": ALFONSI_H_MART, "activation": ALFONSI_ACTIVATION, "feature_scale": ALFONSI_FEATURE_SCALE, "ridge": ALFONSI_RIDGE}

def evaluate_alfonsi(method, payoff, martingale_set, policy_set, price_set, basis_seed):
    common = dict(payoff=payoff, strike_price=STRIKE, interest_rate=RATE, dividend=DIVIDEND, maturity=MATURITY, exercise_dates=EXERCISE_DATES, subintervals=ALFONSI_SUBINTERVALS, martingale_set=martingale_set, policy_set=policy_set, price_set=price_set)
    if method == "alfonsi-standard": return american_AL_PRIMAL_DUAL_standard_local(**common, P=ALFONSI_CELLS_PER_DIMENSION, max_deg=ALFONSI_DEGREE, ridge_alpha=ALFONSI_RIDGE)
    return american_AL_PRIMAL_DUAL(**common, activation=ACTIVATIONS[ALFONSI_ACTIVATION], nneurons_mart=ALFONSI_H_MART, nneurons_reg=ALFONSI_H_REG, std_basis_mart=ALFONSI_FEATURE_SCALE, std_basis_reg=ALFONSI_FEATURE_SCALE, basis_seed=basis_seed, ridge_alpha=ALFONSI_RIDGE)

def run_alfonsi_experiments():
    rows = []
    for dimension in ALFONSI_DIMENSIONS:
        local_coefficients = ALFONSI_CELLS_PER_DIMENSION**dimension * dimension
        for initial_price in ALFONSI_INITIAL_PRICES:
            for seed in ALFONSI_SEEDS:
                runnable = []
                for method in SELECTED_ALFONSI_METHODS:
                    if method == "alfonsi-standard" and local_coefficients > MAX_LOCAL_COEFFICIENTS:
                        for payoff_name in ALFONSI_PAYOFFS: rows.append({**alfonsi_row(method, payoff_name, dimension, initial_price, seed), "status": "skipped: local basis too large"})
                    else: runnable.append(method)
                if not runnable: continue
                martingale_set = simulate(ALFONSI_MARTINGALE_PATHS, dimension, initial_price, seed, ALFONSI_ASYMMETRIC_VOLATILITY, ALFONSI_SUBINTERVALS)
                policy_set = simulate(ALFONSI_POLICY_PATHS, dimension, initial_price, seed + 1, ALFONSI_ASYMMETRIC_VOLATILITY, ALFONSI_SUBINTERVALS)
                price_set = simulate(ALFONSI_PRICING_PATHS, dimension, initial_price, seed + 2, ALFONSI_ASYMMETRIC_VOLATILITY, ALFONSI_SUBINTERVALS)
                for payoff_name in ALFONSI_PAYOFFS:
                    payoff = PAYOFFS[payoff_name]
                    for method in runnable:
                        started = time.perf_counter()
                        lower, lower_se, upper, upper_se, diagnostics = evaluate_alfonsi(method, payoff, martingale_set, policy_set, price_set, BASIS_SEED_OFFSET + seed)
                        rows.append({**alfonsi_row(method, payoff_name, dimension, initial_price, seed), "lower_bound": lower, "lower_standard_error": lower_se, "upper_bound": upper, "upper_standard_error": upper_se, **diagnostics, "duration_seconds": time.perf_counter() - started, "status": "ok"})
                        print(f"Completed {method}/{payoff_name}: d={dimension}, S0={initial_price:g}, seed={seed}")
                del martingale_set, policy_set, price_set; gc.collect()
    result = pd.DataFrame(rows); ALFONSI_OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True); result.to_csv(ALFONSI_OUTPUT_FILE, index=False)
    return result

## Run experiments and display results

In [ ]:
lsmc_hk_results = run_lsmc_hk_experiments() if RUN_LSMC_HK else pd.DataFrame()
alfonsi_results = run_alfonsi_experiments() if RUN_ALFONSI else pd.DataFrame()

lsmc_hk_summary = summarize_results(lsmc_hk_results)
alfonsi_summary = summarize_results(alfonsi_results)
comparison_results = pd.concat([lsmc_hk_summary, alfonsi_summary], ignore_index=True)
LSMC_HK_SUMMARY_FILE.parent.mkdir(parents=True, exist_ok=True)
lsmc_hk_summary.to_csv(LSMC_HK_SUMMARY_FILE, index=False)
ALFONSI_SUMMARY_FILE.parent.mkdir(parents=True, exist_ok=True)
alfonsi_summary.to_csv(ALFONSI_SUMMARY_FILE, index=False)
COMPARISON_OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
comparison_results.to_csv(COMPARISON_OUTPUT_FILE, index=False)

if RUN_LSMC_HK: 
	print(f"LSMC/HK raw results: {LSMC_HK_OUTPUT_FILE}; summary: {LSMC_HK_SUMMARY_FILE}")
	display(lsmc_hk_summary)
if RUN_ALFONSI: 
	print(f"Alfonsi raw results: {ALFONSI_OUTPUT_FILE}; summary: {ALFONSI_SUMMARY_FILE}")
	display(alfonsi_summary)
print(f"Combined summary saved to {COMPARISON_OUTPUT_FILE}"); display(comparison_results)

Completed lsmc-standard/max-call: d=5, S0=90, seed=101
Completed lsmc-random/max-call: d=5, S0=90, seed=101
Completed hk-standard/max-call: d=5, S0=90, seed=101
Completed hk-random/max-call: d=5, S0=90, seed=101
Completed lsmc-standard/max-call: d=5, S0=90, seed=202
Completed lsmc-random/max-call: d=5, S0=90, seed=202
Completed hk-standard/max-call: d=5, S0=90, seed=202
